In [25]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json

In [26]:
# Save data sets with ordered columns
rng = np.random.default_rng(1412)
mu = [0]*4
Sigma0 = [[1.0, 0.5, 0.0, 0.0],
 [0.5, 1.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.4],
 [0.0, 0.0, 0.4, 0.0]]
Sigma1 = [[1.0, 0.8, 0.0, 0.0],
 [0.8, 1.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, -0.2],
 [0.0, 0.0, -0.2, 0.0]]
y = rng.choice([0,1],1000,replace=True,p=[0.5,0.5])
data = np.zeros((1000,4))
data[y==0,] = rng.multivariate_normal(mu, Sigma0, size=1000-y.sum())
data[y==1,] = rng.multivariate_normal(mu, Sigma1, size=y.sum())
data = pd.DataFrame(data, columns=[f"x{i}" for i in range(1,4+1)])
data["y"] = y

In [27]:
data.head()

,x1,x2,x3,x4,y
0,-0.595710,-1.656312,0.580359,0.888136,0
1,1.984155,1.621420,-0.027460,0.000716,1
2,-1.222167,-0.707392,-1.157504,-0.358193,0
3,-0.587291,-0.889086,0.083253,0.135477,1
4,0.416376,0.485841,0.037571,0.217626,1


In [28]:
data.to_csv("../data/test/test.csv", index=False)

In [29]:
# Create json files
def df_to_schema(df, categorical_cols):
    columns = []
    for col in df.columns:
        if col in categorical_cols:
            categories = df[col].astype(str).unique().tolist()
            columns.append({"name": col,
                            "type": "Categorical",
                            "size": len(categories),
                            "i2s": categories})
        else:
            columns.append({"name": col,
                            "type": "Float",
                            "min": float(df[col].min()),
                            "max": float(df[col].max())})
    return {"columns": columns}

# Save json files
with open("../data/test/test.json", "w") as file:
    json.dump(df_to_schema(data, categorical_cols=["y"]), file, indent=2)